# Matrix Factorization (Gradient Descent)
- Đánh giá bằng RMSE trên tập test

In [ ]:
import numpy as np
import pandas as pd

: 

## 1) Đọc dữ liệu MovieLens 100K (ua.base, ua.test)

In [ ]:
train_df = pd.read_csv('../ml-100k/ua.base', sep='\t', names=['user_id', 'movie_id', 'rating', 'ts'])
test_df = pd.read_csv('../ml-100k/ua.test', sep='\t', names=['user_id', 'movie_id', 'rating', 'ts'])

train = train_df[['user_id', 'movie_id', 'rating']].values.astype(np.float64)
test = test_df[['user_id', 'movie_id', 'rating']].values.astype(np.float64)

# Chuyển về index bắt đầu từ 0
train[:, :2] -= 1
test[:, :2] -= 1

train[:5]

: 

## 2) Mô hình MF tối giản

In [ ]:
class MFSimple:
    def __init__(self, data, K=10, lam=0.1, lr=0.75, max_iter=50, print_every=10):
        self.Y_raw = data
        self.K = K
        self.lam = lam
        self.lr = lr
        self.max_iter = max_iter
        self.print_every = print_every

        self.n_users = int(np.max(data[:, 0])) + 1
        self.n_items = int(np.max(data[:, 1])) + 1
        self.n_ratings = data.shape[0]

        self.X = np.random.randn(self.n_items, K) * 0.1
        self.W = np.random.randn(K, self.n_users) * 0.1

        self.Y = self.Y_raw.copy()
        self.mu = np.zeros(self.n_users)

    def normalize_by_user(self):
        users = self.Y_raw[:, 0].astype(np.int32)
        for u in range(self.n_users):
            ids = np.where(users == u)[0]
            if len(ids) == 0:
                continue
            m = np.mean(self.Y_raw[ids, 2])
            self.mu[u] = m
            self.Y[ids, 2] = self.Y_raw[ids, 2] - m

    def get_items_rated_by_user(self, u):
        ids = np.where(self.Y[:, 0] == u)[0]
        item_ids = self.Y[ids, 1].astype(np.int32)
        ratings = self.Y[ids, 2]
        return item_ids, ratings

    def get_users_rate_item(self, i):
        ids = np.where(self.Y[:, 1] == i)[0]
        user_ids = self.Y[ids, 0].astype(np.int32)
        ratings = self.Y[ids, 2]
        return user_ids, ratings

    def update_X(self):
        for i in range(self.n_items):
            user_ids, ratings = self.get_users_rate_item(i)
            if len(user_ids) == 0:
                continue
            Wi = self.W[:, user_ids]
            grad = -((ratings - self.X[i, :].dot(Wi)).dot(Wi.T)) / self.n_ratings + self.lam * self.X[i, :]
            self.X[i, :] -= self.lr * grad

    def update_W(self):
        for u in range(self.n_users):
            item_ids, ratings = self.get_items_rated_by_user(u)
            if len(item_ids) == 0:
                continue
            Xu = self.X[item_ids, :]
            grad = -Xu.T.dot(ratings - Xu.dot(self.W[:, u])) / self.n_ratings + self.lam * self.W[:, u]
            self.W[:, u] -= self.lr * grad

    def pred(self, u, i):
        u = int(u)
        i = int(i)
        p = self.X[i, :].dot(self.W[:, u]) + self.mu[u]
        return np.clip(p, 0, 5)

    def rmse(self, data):
        se = 0.0
        for n in range(data.shape[0]):
            u, i, r = data[n]
            p = self.pred(u, i)
            se += (p - r) ** 2
        return np.sqrt(se / data.shape[0])

    def fit(self):
        self.normalize_by_user()
        for it in range(1, self.max_iter + 1):
            self.update_X()
            self.update_W()
            if it % self.print_every == 0:
                print(f'iter={it}, RMSE_train={self.rmse(self.Y_raw):.4f}')

: 

## 3) Huấn luyện nhanh và đánh giá

In [ ]:
np.random.seed(42)
model = MFSimple(train, K=10, lam=0.1, lr=0.75, max_iter=30, print_every=10)
model.fit()

rmse_test = model.rmse(test)
print('RMSE test =', round(rmse_test, 4))